In [3]:
import pandas as pd

from google.colab import drive
import pandas as pd
drive.mount('/content/drive')
df=pd.read_csv("/content/drive/My Drive/Colab Notebooks/CarPrice_dataset.csv")
df.head()



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,1,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0
1,2,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0
2,3,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0
3,4,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0
4,5,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0


In [4]:
df.isnull().sum()

,0
car_ID,0
symboling,0
CarName,0
fueltype,0
aspiration,0
doornumber,0
carbody,0
drivewheel,0
enginelocation,0
wheelbase,0


# Data cleaning

In [6]:
from pandas.core.arrays import categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer

x=df.drop(columns='price')
y=df['price']

categorical_cols=x.select_dtypes(include=['object']).columns.tolist()
numerical_cols=x.select_dtypes(include=['int64','float64']).columns.tolist()


preprocessor=ColumnTransformer(
    transformers=[
        ('num',StandardScaler(),numerical_cols),
        ('car',OneHotEncoder(handle_unknown='ignore'),categorical_cols)
    ]
)


x_processed=preprocessor.fit_transform(x)

x_train,x_test,y_train,y_test=train_test_split(x_processed,y,test_size=0.2,random_state=42)



# Model Building

In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential([
    
    Dense(512, activation='relu', input_shape=(x_train.shape[1],)),  # Hidden Layer 1
    Dense(256, activation='relu'),
    Dense(128, activation='relu'),  
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1)                                                         # Output Layer
])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │       102,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 277,505 (1.06 MB)

 Trainable params: 277,505 (1.06 MB)

 Non-trainable params: 0 (0.00 B)

# Train the model

In [9]:
model.compile(
    optimizer='adam',
    loss='mean_absolute_error',
    metrics=['mean_absolute_error','mean_squared_error']
)


In [10]:
history = model.fit(
    x_train,
    y_train,
    epochs=90,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)


Epoch 1/90
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 624ms/step - loss: 12367.0664 - mean_absolute_error: 12367.0664 - mean_squared_error: 201130256.0000 - val_loss: 15080.2803 - val_mean_absolute_error: 15080.2803 - val_mean_squared_error: 317853344.0000
Epoch 2/90
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - loss: 13198.5449 - mean_absolute_error: 13198.5449 - mean_squared_error: 231252144.0000 - val_loss: 15071.7227 - val_mean_absolute_error: 15071.7227 - val_mean_squared_error: 317575072.0000
Epoch 3/90
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - loss: 12491.3301 - mean_absolute_error: 12491.3301 - mean_squared_error: 206897600.0000 - val_loss: 15047.6143 - val_mean_absolute_error: 15047.6143 - val_mean_squared_error: 316806688.0000
Epoch 4/90
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - loss: 12700.9844 - mean_absolute_error: 12700.9844 - mean_squared_error: 211839984.0000 - val_loss: 14983.9414 - val_mean_absolute_error: 14983.9414 - val_mean_squared_error: 314802560.0000
Epoch 5/90
5/5 ━━━━━━━━━━━━━━━━━━━━

In [12]:
#model Evaluation
from sklearn.metrics import *
y_pred = model.predict(x_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Mean Absolute Error:", mae)
print("Mean Squared Error:", mse)
print("R² Score:", r2)

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 547ms/step
Mean Absolute Error: 1908.2303613757622
Mean Squared Error: 8823206.433961865
R² Score: 0.8882346460739317


In [18]:
new_data=x.iloc[[0]]
new_data_precessed=preprocessor.transform(new_data)
predicted_price = model.predict(new_data_precessed)
print("Predicted Car Price: ₹", int(predicted_price[0][0]))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 416ms/step
Predicted Car Price: ₹ 13620
